In [7]:
import pandas as pd
import requests
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [8]:
def download_pdb(pdb_id: str, output_dir: str = "./data/pdbs"):
    pdb_id = pdb_id.lower()
    output_path = os.path.join(output_dir, f"{pdb_id}.pdb")

    if os.path.exists(output_path):
        return "EXISTE"

    try:
        response = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=15)
        response.raise_for_status()
        with open(output_path, 'w') as f:
            f.write(response.text)
        return "SUCESSO"
    except requests.exceptions.RequestException:
        return "FALHA"

In [9]:
path_rfam = "/home/andre/BioGraph.AI/src/data/Rfam/Rfam.pdb" 
path_pdbs = "/home/andre/BioGraph.AI/src/data/pdbs"
os.makedirs(path_pdbs, exist_ok=True)

column_names = [
    'rfam_acc', 'pdb_id', 'chain', 'pdb_start', 'pdb_end', 'bit_score', 
    'evalue_score', 'cm_start', 'cm_end', 'hex_colour'
]

mapping_df = pd.read_csv(
    path_rfam, sep=r'\s+', header=None,
    names=column_names, comment='#'
)

unique_pdb_ids = mapping_df['pdb_id'].unique()

success_count = 0
exist_count = 0
failed_ids = []

print(f"Iniciando o download de {len(unique_pdb_ids)} PDBs únicos...")

with ThreadPoolExecutor(max_workers=8) as executor:

    futures = {executor.submit(download_pdb, pdb_id, output_dir=path_pdbs): pdb_id for pdb_id in unique_pdb_ids}

    for future in tqdm(as_completed(futures), total=len(unique_pdb_ids), desc="Baixando PDBs"):
        pdb_id_original = futures[future]
        try:
            status = future.result()
            
            if status == "SUCESSO":
                success_count += 1
            elif status == "EXISTE":
                exist_count += 1
            else:
                failed_ids.append(pdb_id_original)
        except Exception as e:
            print(f"Erro ao processar {pdb_id_original}: {e}")
            failed_ids.append(pdb_id_original)

print("\n--- RESUMO DO DOWNLOAD ---")
print(f"Sucesso: {success_count} novos arquivos baixados.")
print(f"Já existentes: {exist_count} arquivos pulados.")
print(f"Falhas: {len(failed_ids)} PDB IDs não foram baixados.")
if failed_ids:
    print(f"IDs que falharam: {sorted(failed_ids)}")

Iniciando o download de 3122 PDBs únicos...


Baixando PDBs: 100%|██████████| 3122/3122 [06:00<00:00,  8.67it/s] 


--- RESUMO DO DOWNLOAD ---
Sucesso: 0 novos arquivos baixados.
Já existentes: 1563 arquivos pulados.
Falhas: 1559 PDB IDs não foram baixados.
IDs que falharam: ['1vvj', '1vy4', '1vy5', '1vy6', '1vy7', '3j6b', '3j6x', '3j6y', '3j77', '3j78', '3j79', '3j7o', '3j7p', '3j7q', '3j7r', '3j92', '3j9m', '3j9w', '3j9y', '3j9z', '3ja1', '3jag', '3jah', '3jai', '3jaj', '3jan', '3jbn', '3jbo', '3jbp', '3jbu', '3jbv', '3jcd', '3jce', '3jcj', '3jcn', '3jcs', '3jct', '4bts', '4d5y', '4d67', '4l47', '4l71', '4lel', '4lfz', '4lnt', '4lsk', '4lt8', '4p6f', '4p70', '4tua', '4tub', '4tuc', '4tud', '4tue', '4u1u', '4u1v', '4u20', '4u24', '4u25', '4u26', '4u27', '4u3m', '4u3n', '4u3u', '4u4n', '4u4o', '4u4q', '4u4r', '4u4u', '4u4y', '4u4z', '4u50', '4u51', '4u52', '4u53', '4u55', '4u56', '4u6f', '4ug0', '4ujc', '4ujd', '4uje', '4v3p', '4v42', '4v47', '4v48', '4v49', '4v4a', '4v4b', '4v4g', '4v4h', '4v4i', '4v4j', '4v4n', '4v4p', '4v4q', '4v4r', '4v4s', '4v4t', '4v4v', '4v4w', '4v4x', '4v4y', '4v4z', '4v50'

Posição (Colunas)	Campo	Exemplo na sua Linha	Descrição


1 - 6	Record Name	ATOM	Identifica que a linha descreve um átomo em uma molécula padrão.


7 - 11	Atom Serial Number	509	Um número de série único para cada átomo no arquivo.


13 - 16	Atom Name	N6	O nome do átomo dentro do seu resíduo (ex: N6 de Adenina, P de Fósforo no esqueleto).


17	Alt. Location	(vazio)	Usado se um átomo pode existir em múltiplas posições.


18 - 20	Residue Name	A	O nome do resíduo (nucleotídeo ou aminoácido). No seu caso: A=Adenina, U=Uracila.


22	Chain Identifier	R	A letra ou número que identifica a cadeia da molécula. Útil quando há várias moléculas no arquivo.


23 - 26	Residue Seq. Num.	624	O número sequencial do resíduo dentro daquela cadeia.


27	Insertion Code	(vazio)	Usado para inserções na sequência.


31 - 38	X Coordinate	112.736	A coordenada X da posição do átomo no espaço, em Ångströms (Å).


39 - 46	Y Coordinate	51.320	A coordenada Y da posição do átomo no espaço, em Ångströms (Å).


47 - 54	Z Coordinate	-15.288	A coordenada Z da posição do átomo no espaço, em Ångströms (Å).


55 - 60	Occupancy	1.00	Um fator que indica a proporção de tempo que o átomo é encontrado naquela posição (geralmente 1.00).


61 - 66	Temp. Factor	33.26	Também chamado de B-factor, indica o quão "desordenado" ou vibrátil é o átomo. Valores mais baixos indicam uma posição mais estável e bem definida.


77 - 78	Element Symbol	N	O símbolo do elemento químico (N para Nitrogênio, C para Carbono, P para Fósforo, etc.).


79 - 80	Charge	(vazio)	A carga do átomo.

ModuleNotFoundError: No module named 'BioPython'